# Conway's Game of Life

A cellular automaton devised by mathematician John Conway in 1970. Each cell on a 2D grid is either **alive** or **dead**, and evolves according to four simple rules based on its eight neighbors:

1. **Underpopulation**: A live cell with fewer than 2 live neighbors dies.
2. **Survival**: A live cell with 2 or 3 live neighbors survives.
3. **Overpopulation**: A live cell with more than 3 live neighbors dies.
4. **Reproduction**: A dead cell with exactly 3 live neighbors becomes alive.

This notebook uses the Minkowski ECS Python bridge to simulate Life on a 64x64 grid. Each cell is an ECS entity with a `CellState` component, and the `life_step` reducer applies Conway's rules each generation.

In [ ]:
import random

import matplotlib.pyplot as plt
import minkowski_py as mk
from IPython.display import clear_output

WIDTH, HEIGHT = 64, 64
GENERATIONS = 200

## Initialize the Grid

Spawn `WIDTH × HEIGHT` cells in **row-major order** — the `life_step` reducer uses this ordering to compute neighbor indices. Entity 0 = cell (0,0), entity 1 = cell (1,0), ..., so `index = x + y * WIDTH`.

In [ ]:
world = mk.World()
registry = mk.ReducerRegistry(world)

random.seed(42)
alive_list = [random.random() < 0.35 for _ in range(WIDTH * HEIGHT)]
ids = world.spawn_batch("CellState", {"alive": alive_list})
print(f"Spawned {len(ids)} cells ({sum(alive_list)} alive)")

## Helper: DataFrame → Grid

Convert the query DataFrame to a NumPy array for visualization. We sort by `entity_id` to preserve row-major order.

In [ ]:
def to_grid(world):
    df = world.query("CellState").sort("entity_id")
    return df["alive"].to_numpy().reshape(HEIGHT, WIDTH)


grid = to_grid(world)
fig, ax = plt.subplots(figsize=(7, 7))
ax.imshow(grid, cmap="binary_r", interpolation="nearest")
ax.set_title(f"Generation 0 — {grid.sum()} alive")
plt.tight_layout()
plt.show()

## Run Simulation

Each call to `life_step` snapshots the current state, counts neighbors via index arithmetic (toroidal wrapping), and applies Conway's rules.

In [ ]:
frames = [grid.copy()]

for gen in range(1, GENERATIONS + 1):
    registry.run("life_step", world, width=WIDTH, height=HEIGHT)
    if gen % 5 == 0:
        frames.append(to_grid(world))
    if gen % 50 == 0:
        g = to_grid(world)
        print(f"Generation {gen}: {g.sum()} alive")

print(f"Done — {len(frames)} frames captured")

## Final State

In [ ]:
fig, ax = plt.subplots(figsize=(7, 7))
ax.imshow(frames[-1], cmap="binary_r", interpolation="nearest")
ax.set_title(f"Generation {GENERATIONS} — {frames[-1].sum()} alive")
plt.tight_layout()
plt.show()

## Animation

In [ ]:
fig, ax = plt.subplots(figsize=(7, 7))
im = ax.imshow(frames[0], cmap="binary_r", interpolation="nearest")

for i, frame in enumerate(frames):
    im.set_data(frame)
    ax.set_title(f"Generation {i * 5} — {frame.sum()} alive")
    clear_output(wait=True)
    plt.pause(0.05)
    display(fig)

plt.close(fig)

## Classic Pattern: R-pentomino

The R-pentomino is a 5-cell methuselah that takes 1103 generations to stabilize. Let's watch its early evolution on a fresh grid.

In [ ]:
world2 = mk.World()
registry2 = mk.ReducerRegistry(world2)

cells = [False] * (WIDTH * HEIGHT)
cx, cy = WIDTH // 2, HEIGHT // 2
# R-pentomino: .##  / ##.  / .#.
for dx, dy in [(1, 0), (2, 0), (0, 1), (1, 1), (1, 2)]:
    cells[(cx + dx) + (cy + dy) * WIDTH] = True

world2.spawn_batch("CellState", {"alive": cells})

# Run 300 generations, capture every 3rd
r_frames = []
for gen in range(301):
    if gen % 3 == 0:
        df = world2.query("CellState").sort("entity_id")
        r_frames.append(df["alive"].to_numpy().reshape(HEIGHT, WIDTH))
    if gen < 300:
        registry2.run("life_step", world2, width=WIDTH, height=HEIGHT)

# Animate
fig, ax = plt.subplots(figsize=(7, 7))
im = ax.imshow(r_frames[0], cmap="binary_r", interpolation="nearest")

for i, frame in enumerate(r_frames):
    im.set_data(frame)
    ax.set_title(f"R-pentomino — Generation {i * 3} — {frame.sum()} alive")
    clear_output(wait=True)
    plt.pause(0.03)
    display(fig)

plt.close(fig)

## Explore

- Try different initial densities (20%, 50%, 70%) — how does the stable population change?
- Implement a [Gosper Glider Gun](https://conwaylife.com/wiki/Gosper_glider_gun) for infinite growth
- Try larger grids: `WIDTH, HEIGHT = 128, 128`
- Compare generation counts to stable/oscillating states